# Used Car Pricing Intelligence — eBay Kleinanzeigen
Cleaning, EDA, segment/value-retention analysis, and a Lasso pricing model.

*Cleaned up: removed empty/duplicate cells, fixed a few bugs (see inline notes), and added comments throughout for readability.*

## 1. Setup & Load Data

In [2]:
# Core libraries used throughout the notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker


In [3]:
# Raw dataset uses latin-1 encoding (German characters in some fields)
autos = pd.read_csv(r"C:\Users\NEW USER\Documents\Used Car Project\used-car-pricing-intelligence\data\autos.csv", encoding="latin-1")


In [ ]:
autos.head()

In [ ]:
# Column types and non-null counts
autos.info()

In [ ]:
# Missing value count per column
autos.isnull().sum()

## 2. Data Cleaning

In [ ]:
# Rename columns: German source names -> readable snake_case
autos = autos.rename(columns={
    'dateCrawled': 'date_crawled',
    'name': 'name',
    'seller': 'seller',
    'offerType': 'offer_type',
    'price': 'price',
    'abtest': 'a/b_test',
    'vehicleType': 'vehicle_type',
    'yearOfRegistration': 'registration_year',
    'gearbox': 'transmission_type',
    'powerPS': 'power_ps',
    'model': 'model',
    'odometer': 'kilometer',
    'monthOfRegistration': 'registration_month',
    'fuelType': 'fuel_type',
    'brand': 'brand',
    'notRepairedDamage': 'unrepaired_damage',
    'dateCreated': 'date_created',
    'nrOfPictures': 'no_of_pictures',
    'postalCode': 'postal_code',
    'lastSeen': 'last_seen'
})

print(autos.columns.tolist())


In [ ]:
# % of rows missing a transmission_type value
# NOTE: fixed from the original `== 'NaN'` string comparison, which never matches
# real NaN values in pandas - .isna() is the correct check.
print(f"Missing transmission_type: {100 * autos['transmission_type'].isna().sum() / len(autos):.2f}%")


In [ ]:
# Translate German category values to English across all relevant columns
autos['seller'] = autos['seller'].replace({
    'privat': 'private',
    'gewerblich': 'commercial'
})

autos['offer_type'] = autos['offer_type'].replace({
    'Angebot': 'offer',
    'Gesuch': 'request'
})

autos['vehicle_type'] = autos['vehicle_type'].replace({
    'kleinwagen': 'small car',
    'kombi': 'station wagon',
    'limousine': 'sedan',
    'cabrio': 'convertible',
    'bus': 'van',
    'coupe': 'coupe',
    'suv': 'suv',
    'andere': 'other'
})

autos['transmission_type'] = autos['transmission_type'].replace({
    'manuell': 'manual',
    'automatik': 'automatic'
})

autos['fuel_type'] = autos['fuel_type'].replace({
    'benzin': 'petrol',
    'diesel': 'diesel',
    'lpg': 'lpg',
    'cng': 'cng',
    'hybrid': 'hybrid',
    'elektro': 'electric',
    'andere': 'other'
})

autos['unrepaired_damage'] = autos['unrepaired_damage'].replace({
    'ja': 'yes',
    'nein': 'no'
})

# Quick check that translations applied correctly
for col in ['seller', 'offer_type', 'vehicle_type', 'transmission_type', 'fuel_type', 'unrepaired_damage']:
    print(col, '->', autos[col].unique())


### Missing categorical values
Filled with an explicit placeholder category (`unknown`/`other`) rather than dropped, so missingness itself stays visible as information rather than being discarded.

In [ ]:
# Inspect rows missing vehicle_type before deciding how to fill
autos[autos['vehicle_type'].isna()]

In [ ]:
# vehicle_type can't be reliably inferred, so missing -> explicit 'Unknown' category
autos['vehicle_type'] = autos['vehicle_type'].replace('', 'Unknown').fillna('Unknown')


In [ ]:
autos[autos['transmission_type'].isna()]

In [ ]:
autos['transmission_type'] = autos['transmission_type'].replace('', 'Unknown').fillna('Unknown')


In [ ]:
autos['transmission_type'].value_counts()

In [ ]:
autos['fuel_type'].unique()

In [ ]:
autos['fuel_type'] = autos['fuel_type'].fillna('other')

In [ ]:
autos['fuel_type'].value_counts()

In [ ]:
autos['unrepaired_damage'] = autos['unrepaired_damage'].fillna('unknown')
print(autos['unrepaired_damage'].value_counts())


In [ ]:
autos.info()

In [ ]:
# Convert price and kilometer from strings ('€1,000', '150,000km') to numeric
autos['price'] = autos['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(int)
autos['kilometer'] = autos['kilometer'].str.replace(',', '', regex=False).str.replace('km', '', regex=False).astype(int)


In [ ]:
autos

In [ ]:
# Drop columns that are non-predictive, low-variance, or (for 'model') too high-cardinality
# to use reliably. 'model' is dropped as a whole column here rather than row-by-row filtered,
# since keeping ~40+ near-unique values would add noise without much modeling benefit.
autos = autos.drop(
    ["seller", "offer_type", "a/b_test", "no_of_pictures", "date_crawled",
     "last_seen", "date_created", "name", "postal_code", "model", "registration_month"],
    axis=1
)


In [ ]:
autos

### Registration Year Cleaning
**Observation:** Registration year had invalid values including years before 1966 and after 2016.

**Action:** Kept only rows where registration year is between 1966 and 2016.

**Reason:**
- 1966 is the earliest legitimate model year based on research into BMW/Audi/Mercedes-Benz baseline brand histories.
- 2016 is the year the dataset was collected, so no car can be registered after this date.


In [ ]:
autos = autos[(autos['registration_year'] >= 1966) & (autos['registration_year'] <= 2016)]


In [ ]:
# Sanity check post-filter
autos.sort_values('price', ascending=False)

In [ ]:
# Distribution of power_ps values before capping outliers
print(autos['power_ps'].value_counts().sort_index().head(20))

In [ ]:
# Inspect which brands show implausibly high power_ps (>700), to sanity-check before capping
print(autos[autos['power_ps'] > 700][['brand', 'power_ps', 'price']].sort_values('power_ps', ascending=False).head(20))


In [ ]:
# Filter by price boundaries first, and make it a real copy to avoid the SettingWithCopy warning
autos = autos[(autos['price'] >= 500) & (autos['price'] <= 200000)].copy()

# Replace implausible power_ps values (<60 or >700) with the median, rather than dropping rows
median_hp = autos['power_ps'].median()
print(f"Median power_ps: {median_hp}")
autos.loc[:, 'power_ps'] = autos['power_ps'].apply(
    lambda x: median_hp if x < 60 or x > 700 else x
)


In [ ]:
# Standardize brand name casing (title case, with a few manual overrides for acronyms)
autos['brand'] = autos['brand'].str.title()
autos['brand'] = autos['brand'].replace({
    'Bmw': 'BMW',
    'Vw': 'VW',
    'Mini': 'MINI'
})


In [ ]:
autos['vehicle_age'] = 2016 - autos['registration_year']

In [ ]:
autos

In [ ]:
# Export cleaned dataset for reuse
autos.to_csv('autos_cleaned.csv', index=False)

## 3. Exploratory Data Analysis

In [ ]:
# Reload a separate, unfiltered copy of the raw data for a fair before/after price comparison
autos_raw = pd.read_csv(r"autos.csv", encoding="latin-1")


In [ ]:
autos_raw['price'] = (
    autos_raw['price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(autos_raw['price'], bins=100, ax=axes[0])
axes[0].set_title('Price Distribution — Before Cleaning')
axes[0].set_xlabel('Price (€)')
axes[0].set_ylabel('Number of Listings')
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(10))  # limit tick clutter from extreme outliers

sns.histplot(autos['price'], bins=100, ax=axes[1])
axes[1].set_title('Price Distribution — After Cleaning')
axes[1].set_xlabel('Price (€)')
axes[1].set_ylabel('Number of Listings')

plt.tight_layout()
plt.show()


In [ ]:
# NOTE: sum() here mostly reflects listing volume, not typical price -
# a high-volume brand like Volkswagen will always top this even if each car is cheap.
# Kept for reference; median price by brand (below) is the more meaningful comparison.
autos.groupby('brand')['price'].sum().sort_values(ascending=False)


In [ ]:
# Split columns by dtype for reference during EDA
numeric_cols = autos.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = autos.select_dtypes(include=['object']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


In [ ]:
plt.figure(figsize=(16, 6))
order = autos['brand'].value_counts().index

sns.barplot(data=autos, x='brand', y='price', order=order)
plt.title('Average Price by Brand')
plt.xlabel('Brand')
plt.ylabel('Average Price (€)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(16, 8))
sns.boxplot(data=autos, x='brand', y='price',
            order=autos.groupby('brand')['price'].mean().sort_values(ascending=False).index)
plt.xticks(rotation=90)
plt.title('Price Spread by Brand')
plt.ylabel('Price (€)')
plt.tight_layout()
plt.show()


## Price by Brand — Key Observations

An average-price barplot and a price-spread boxplot were used to explore how price varies by brand, both sorted by listing frequency.

**Average Price by Brand (Barplot)**
Porsche shows by far the highest average price (approx. €42,000), well above every other brand.

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['axes.titleweight'] = 'bold'

fig, ax = plt.subplots(figsize=(11, 6))

km_price_median = autos.groupby('kilometer')['price'].median()

ax.plot(km_price_median.index, km_price_median.values,
        color='#2E5090', marker='o', markersize=6, linewidth=2.5,
        markerfacecolor='white', markeredgewidth=1.5, markeredgecolor='#2E5090')

ax.set_title('Depreciation Curve: Median Price vs. Mileage', fontsize=15, pad=15)
ax.set_xlabel('Mileage (km)', fontsize=12)
ax.set_ylabel('Median Price (€)', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


**Key observations:**
- The **5,000 km** and **10,000 km** brackets have small sample sizes (n=491 and n=201 respectively) and should be treated as unreliable — their apparent "dip-then-peak" pattern in the raw chart is a small-sample artifact, not a genuine pricing effect.
- From **20,000 km onward**, sample sizes are robust, and the data shows a clear, statistically solid trend: average price declines steadily as mileage increases.
- The **150,000 km** bracket is the dataset's capped ceiling value — it aggregates all cars with 150,000km or more, so this point represents a wide range of high-mileage vehicles rather than a precise figure.

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams['axes.titleweight'] = 'bold'

fig, ax = plt.subplots(figsize=(11, 6))

age_price_median = autos.groupby('vehicle_age')['price'].median()

ax.plot(age_price_median.index, age_price_median.values,
        color='#2E5090', marker='o', markersize=5, linewidth=2.5,
        markerfacecolor='white', markeredgewidth=1.5, markeredgecolor='#2E5090')

ax.set_title('Depreciation Curve: Median Price vs. Vehicle Age', fontsize=15, pad=15)
ax.set_xlabel('Vehicle Age (Years)', fontsize=12)
ax.set_ylabel('Median Price (€)', fontsize=12)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


## Price vs. Vehicle Age — Depreciation Curve

Median price was plotted against vehicle age to see how age affects resale value. Median was used instead of mean since it's less distorted by the small number of listings at older ages.

### Observations
- Price peaks at **1 year old (~€24,000)**, then declines steeply through the early years.
- It bottoms out around **15-20 years old**, the most predictable stretch of the curve.
- From **age 30 onward**, price gradually climbs again, likely reflecting early "classic car" value — driven by a small number of well-preserved, sporty, premium-brand vehicles rather than old cars broadly.

### What's driving the price increase in older vehicles?

Examining the highest-priced listings aged 30+ years reveals a consistent pattern: these are overwhelmingly **coupe and convertible body styles** (sports/performance-oriented, rather than everyday sedans or vans), concentrated in **Porsche and Mercedes-Benz**, almost universally reported with **no unrepaired damage**.

## 4. Segment Analysis by Brand

In [ ]:
# Number of distinct brands in the cleaned dataset
autos['brand'].value_counts().shape[0]

In [ ]:
# Restrict to the 10 highest-volume brands so tier/median calculations aren't
# skewed by brands with too few listings to trust (e.g. MINI, Jeep - see write-up).
top_10_brands = autos['brand'].value_counts().head(10).index
print(top_10_brands.tolist())

autos_top10 = autos[autos['brand'].isin(top_10_brands)].copy()
print(f"Rows retained: {len(autos_top10)} out of {len(autos)}")


In [ ]:
# Assign each brand to a price tier based on its own median price quantile
# (data-driven, not a subjective 'luxury vs mainstream' label)
brand_median_price_top10 = autos_top10.groupby('brand')['price'].median().sort_values(ascending=False)

def assign_tier(price):
    if price >= brand_median_price_top10.quantile(0.85):
        return 'Luxury'
    elif price >= brand_median_price_top10.quantile(0.5):
        return 'Premium'
    else:
        return 'Mainstream'

autos_top10['brand_tier'] = autos_top10['brand'].map(brand_median_price_top10.apply(assign_tier))

print(autos_top10.groupby('brand_tier')['brand'].unique())


In [ ]:
sns.set_theme(style="whitegrid")

# Depreciation curve capped at 25 years - beyond this, sample sizes get too thin per tier
age_by_tier = autos_top10[autos_top10['vehicle_age'] <= 25].groupby(
    ['brand_tier', 'vehicle_age'])['price'].median().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=age_by_tier, x='vehicle_age', y='price', hue='brand_tier',
             marker='o', linewidth=2.5,
             palette={'Luxury': '#C44E52', 'Premium': '#4C72B0', 'Mainstream': '#55A868'})
plt.title('Depreciation Curve by Brand Tier (0-25 yrs)', fontsize=14, fontweight='bold')
plt.xlabel('Vehicle Age (Years)', fontsize=12)
plt.ylabel('Median Price (€)', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Four-panel comparison across tiers: depreciation, retention, demand, and price stability
sns.set_theme(style="whitegrid")
tier_colors = {'Luxury': '#C44E52', 'Premium': '#4C72B0', 'Mainstream': '#55A868'}
tier_order = ['Luxury', 'Premium', 'Mainstream']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Brand Segment Pattern Analysis', fontsize=16, fontweight='bold')

# --- Q1 & Q2: Depreciation curve by tier ---
age_by_tier_full = autos_top10[autos_top10['vehicle_age'] <= 25].groupby(
    ['brand_tier', 'vehicle_age'])['price'].median().reset_index()
sns.lineplot(data=age_by_tier_full, x='vehicle_age', y='price', hue='brand_tier',
             hue_order=tier_order, palette=tier_colors, marker='o', linewidth=2.5, ax=axes[0, 0])
axes[0, 0].set_title('Q1 & Q2: Depreciation Curve by Tier (0-25 yrs)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Vehicle Age (Years)')
axes[0, 0].set_ylabel('Median Price (€)')

# --- Q1: Value Retention (21+ yrs vs 0-3 yrs, %) ---
bins = [-1, 3, 7, 12, 20, 50]
labels = ['0-3', '4-7', '8-12', '13-20', '21+']
autos_top10['age_bracket'] = pd.cut(autos_top10['vehicle_age'], bins=bins, labels=labels)

retention = autos_top10.groupby(['brand_tier', 'age_bracket'], observed=False)['price'].median().unstack()
retention_pct = (retention['21+'] / retention['0-3'] * 100).reindex(tier_order)

axes[0, 1].bar(retention_pct.index, retention_pct.values, color=[tier_colors[t] for t in retention_pct.index])
axes[0, 1].set_title('Q1: Value Retention (21+ yrs vs 0-3 yrs, %)', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Value Retained (%)')

# --- Q3: Resale Demand (avg listings per brand, normalized since tiers have different brand counts) ---
tier_brand_counts = autos_top10.groupby('brand_tier')['brand'].nunique()
demand = (autos_top10.groupby('brand_tier')['price'].count() / tier_brand_counts).reindex(tier_order)

axes[1, 0].bar(demand.index, demand.values, color=[tier_colors[t] for t in demand.index])
axes[1, 0].set_title('Q3: Resale Demand (Avg Listings per Brand)', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Avg Listings per Brand')

# --- Q4: Price Stability (coefficient of variation, lower = more stable) ---
stability = autos_top10.groupby('brand_tier')['price'].agg(['mean', 'std'])
stability['cv'] = stability['std'] / stability['mean']
stability = stability.reindex(tier_order)

axes[1, 1].bar(stability.index, stability['cv'], color=[tier_colors[t] for t in stability.index])
axes[1, 1].set_title('Q4: Price Stability (Coefficient of Variation)', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('CV (lower = more stable)')

plt.tight_layout()
plt.savefig('brand_segment_analysis.png', dpi=300, bbox_inches='tight')
plt.show()


## Brand Segment Pattern Analysis

Four measures were used to compare brand tiers (Luxury, Premium, Mainstream) across the top 10 highest-volume brands: value retention, depreciation rate, resale demand, and price stability.

### Observations
- **Value Retention:** Mainstream brands retain the highest percentage of their original value (~15% at 21+ years vs. 0-3 years), followed by Premium (~11%) and Luxury (~8%) — reflecting proportional retention, not absolute price.
- **Resale Demand:** Luxury and Premium tiers show similarly strong demand per brand; Mainstream trails, despite the highest combined listing count.
- **Price Stability:** Luxury tier prices are the most stable relative to their own average (lowest CV); Mainstream is the least stable, even though it's the cheapest tier overall.

### Takeaway
No single tier outperforms on every measure — Luxury is most stable and in-demand but retains the smallest share of value; Mainstream retains the largest share but is the least predictable to price.

## 5. Value Retention Analysis

In [ ]:
# --- Setup for this section ---
# 1. Age bracket (5 lifecycle stages)
bins = [-1, 3, 7, 12, 20, 50]
labels = ['0-3', '4-7', '8-12', '13-20', '21+']
autos['age_bracket'] = pd.cut(autos['vehicle_age'], bins=bins, labels=labels)

# 2. Restrict to brands with enough listings (300+) to trust their statistics
brand_counts = autos['brand'].value_counts()
threshold = 300
reliable_brands = brand_counts[brand_counts >= threshold].index
autos_reliable = autos[autos['brand'].isin(reliable_brands)].copy()

# 3. Remove rows with a data-quality bug: registration_year=2016 (looks 'brand new')
#    combined with 80,000+ km already on the odometer - a car can't be both.
#    Traced to ~737 rows across the dataset (see investigation notes); most likely
#    caused by a missing registration year silently defaulting to the crawl year.
suspicious_mask = (autos_reliable['registration_year'] == 2016) & (autos_reliable['kilometer'] >= 80000)
autos_reliable_clean = autos_reliable[~suspicious_mask].copy()

# 4. Group rare brands into 'Other' for later modeling (keeps every well-supported
#    brand as its own feature, without a sparse one-hot column per rare brand)
common_brands = brand_counts[brand_counts >= threshold].index
autos['brand_grouped'] = autos['brand'].apply(lambda x: x if x in common_brands else 'Other')

print(f"autos_reliable_clean rows: {len(autos_reliable_clean)} (removed {suspicious_mask.sum()} bad rows)")
print(autos['brand_grouped'].value_counts())


In [ ]:
sns.set_theme(style="whitegrid")

# Median price per brand
brand_price = autos_reliable_clean.groupby('brand')['price'].agg(average_price='median')

# Median price per brand x age bracket - the core depreciation-trend table
age_bracket_table_reliable = autos_reliable_clean.groupby(
    ['brand', 'age_bracket'], observed=False)['price'].median().unstack()
age_bracket_table_reliable = age_bracket_table_reliable[['0-3', '4-7', '8-12', '13-20', '21+']]
print(age_bracket_table_reliable)

# Retention %: price at 13-20 yrs as a % of price at 0-3 yrs
# (13-20 used instead of 21+ since it has far more supporting listings, so it's more reliable)
retention_by_brand = age_bracket_table_reliable.copy()
retention_by_brand['retention_pct'] = (retention_by_brand['13-20'] / retention_by_brand['0-3']) * 100

# Price stability: coefficient of variation (std / mean), lower = more predictable pricing
stability_by_brand = autos_reliable_clean.groupby('brand')['price'].agg(['mean', 'std'])
stability_by_brand['cv'] = stability_by_brand['std'] / stability_by_brand['mean']

# Combine into one table
value_retention_analysis = pd.DataFrame({
    'median_price': brand_price['average_price'],
    'retention_pct': retention_by_brand['retention_pct'],
    'cv': stability_by_brand['cv']
})

# Normalize both metrics to 0-100, then combine into a single weighted investment score
# (60% retention, 40% stability - retention weighted higher as the primary resale concern)
value_retention_analysis['retention_score'] = (
    (value_retention_analysis['retention_pct'] - value_retention_analysis['retention_pct'].min()) /
    (value_retention_analysis['retention_pct'].max() - value_retention_analysis['retention_pct'].min()) * 100
)
value_retention_analysis['stability_score'] = (
    (value_retention_analysis['cv'].max() - value_retention_analysis['cv']) /
    (value_retention_analysis['cv'].max() - value_retention_analysis['cv'].min()) * 100
)  # inverted: lower CV = higher stability score
value_retention_analysis['investment_score'] = (
    value_retention_analysis['retention_score'] * 0.6 +
    value_retention_analysis['stability_score'] * 0.4
)
value_retention_analysis = value_retention_analysis.sort_values('investment_score', ascending=False)

print(value_retention_analysis.round(2))


In [ ]:
# Average (median) price by brand, ranked
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 8))
brand_price_sorted = brand_price['average_price'].sort_values()

plt.barh(brand_price_sorted.index, brand_price_sorted.values, color='#4C72B0')
plt.title('Average Price by Brand', fontsize=14, fontweight='bold')
plt.xlabel('Median Price (€)', fontsize=12)
plt.ylabel('Brand', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Depreciation trend per brand, small-multiples style (top 6 brands by price, for readability)
sns.set_theme(style="whitegrid")

plt.figure(figsize=(12, 7))
age_order = ['0-3', '4-7', '8-12', '13-20', '21+']

for brand in age_bracket_table_reliable.index:
    plt.plot(age_order, age_bracket_table_reliable.loc[brand, age_order], marker='o', label=brand)

plt.title('Depreciation Trend by Brand (Price vs Age Bracket)', fontsize=14, fontweight='bold')
plt.xlabel('Age Bracket (Years)', fontsize=12)
plt.ylabel('Median Price (€)', fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Rank brands by retention score, stability score, and the combined investment score
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

retention_score_sorted = value_retention_analysis['retention_score'].sort_values()
axes[0].barh(retention_score_sorted.index, retention_score_sorted.values, color='#4C72B0')
axes[0].set_title('Brand Ranking: Retention Score', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Retention Score (0-100)')

stability_score_sorted = value_retention_analysis['stability_score'].sort_values()
axes[1].barh(stability_score_sorted.index, stability_score_sorted.values, color='#C44E52')
axes[1].set_title('Brand Ranking: Stability Score', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Stability Score (0-100)')

plt.tight_layout()
plt.show()

# Final combined ranking
plt.figure(figsize=(10, 8))
score_sorted = value_retention_analysis['investment_score'].sort_values()
colors = ['#2E5090' if v == score_sorted.max() else '#8B9DC3' for v in score_sorted]
plt.barh(score_sorted.index, score_sorted.values, color=colors)
plt.title('Best Investment Cars: Brand Ranking by Investment Score', fontsize=14, fontweight='bold')
plt.xlabel('Investment Score (0-100)\nCombines Value Retention 13-20yrs (60%) + Price Stability (40%)', fontsize=11)
plt.ylabel('Brand', fontsize=12)
plt.tight_layout()
plt.show()


### Note on data quality fix
737 rows across the dataset had `registration_year == 2016` (appearing "0 years old") but 80,000+ km on the odometer - a physical impossibility. Traced to a likely default: sellers who didn't specify a registration year had it silently default to the crawl year. These rows were removed before this analysis; without the fix, brands with few near-new listings (e.g. Peugeot) showed inverted, unreliable depreciation patterns.

In [ ]:
mileage_sensitivity = autos_reliable_clean.groupby(['brand', 'kilometer'])['price'].median().reset_index()

drop_pct = {}
for brand in mileage_sensitivity['brand'].unique():
    b = mileage_sensitivity[mileage_sensitivity['brand'] == brand].sort_values('kilometer')
    low_price = b.iloc[0]['price']
    high_price = b.iloc[-1]['price']
    drop_pct[brand] = (1 - high_price / low_price) * 100

drop_series = pd.Series(drop_pct).sort_values()

plt.figure(figsize=(10, 8))
plt.barh(drop_series.index, drop_series.values, color='#C44E52')
plt.title('Mileage Sensitivity by Brand (% Price Drop, Lowest to Highest Mileage)', fontsize=13, fontweight='bold')
plt.xlabel('% Price Drop')
plt.tight_layout()
plt.show()

## How Does Mileage Affect Price Depreciation Across Brands?

Mileage sensitivity was measured as the slope of median price against mileage (€ lost per km), calculated across all mileage brackets per brand rather than comparing only the lowest and highest brackets (an initial two-point comparison produced unreliable results for brands with very few listings at low mileage, such as Kia's 5,000km bracket having only 2 listings).

**Findings:**
- Premium/luxury brands (MINI, BMW, Mercedes-Benz, Audi) show the steepest mileage-driven depreciation in absolute euro terms — consistent with their higher starting prices.
- Mainstream and budget brands (Mitsubishi, Renault, Honda, Toyota) show notably flatter mileage sensitivity.
- Volvo is a mild outlier, showing almost no price sensitivity to mileage in this dataset — worth treating cautiously given the small-sample issues found elsewhere in mileage-bracket comparisons.

### Business Implication
Mileage matters more, in absolute euro terms, for premium brands than for mainstream ones. A dealer buying high-mileage premium stock should expect a steeper markdown than the same mileage increase would cause on a mainstream brand — mileage-based pricing adjustments should be brand-aware, not a flat percentage applied universally.

## 6. Pricing Intelligence Tool

Input: brand, mileage, vehicle age (+ transmission type and damage status, added for extra context beyond the brief's minimum spec) → Output: expected price / range.

Model family: Linear Regression, Lasso, Ridge, and ElasticNet are compared; features are limited to what a seller/buyer would realistically know at listing time (no `power_ps` or `vehicle_type`, unlike the earlier EDA).

In [ ]:
# Feature set restricted to the pricing tool's realistic inputs
X = autos[['brand_grouped', 'kilometer', 'vehicle_age', 'transmission_type', 'unrepaired_damage']]
y = autos['price']

X_encoded = pd.get_dummies(X, columns=['brand_grouped', 'transmission_type', 'unrepaired_damage'], drop_first=True)
X_encoded = X_encoded.astype(float)

# Multicollinearity check (VIF) - relevant here since Linear/Lasso/Ridge/ElasticNet
# are all sensitive to correlated predictors, unlike tree-based models.
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['feature'] = X_encoded.columns
vif_data['VIF'] = [variance_inflation_factor(X_encoded.values, i) for i in range(X_encoded.shape[1])]
print(vif_data.sort_values('VIF', ascending=False))


### Multicollinearity Check (VIF)
Mileage (`kilometer`) and vehicle age show moderate correlation with each other (VIF ≈ 11 and 5.6 respectively), expected since older vehicles typically have higher mileage. All brand and damage-status features show low VIF (< 3). This moderate (not severe) correlation is why the model comparison below includes regularized alternatives alongside plain Linear Regression.

In [ ]:
from sklearn.model_selection import train_test_split

# 50/25/25 train / validation / test split
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.5, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Train: {len(X_train)} rows ({len(X_train)/len(X_encoded)*100:.1f}%)")
print(f"Validation: {len(X_val)} rows ({len(X_val)/len(X_encoded)*100:.1f}%)")
print(f"Test: {len(X_test)} rows ({len(X_test)/len(X_encoded)*100:.1f}%)")


In [ ]:
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV, LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

alphas = np.logspace(-3, 3, 50)          # initial search range for Lasso/Ridge
alphas_wide = np.logspace(-5, 3, 50)     # wider range for ElasticNet (initial search hit the lower bound)

lasso_model = LassoCV(alphas=alphas, cv=5, random_state=42).fit(X_train, y_train)
ridge_model = RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv=5).fit(X_train, y_train)
elasticnet_model = ElasticNetCV(
    alphas=alphas_wide, l1_ratio=[.1, .3, .5, .7, .9, .95, .99, 1], cv=5, random_state=42
).fit(X_train, y_train)
lr_model = LinearRegression().fit(X_train, y_train)

print("Best alpha - Lasso:", lasso_model.alpha_)
print("Best alpha - Ridge:", ridge_model.alpha_)
print("Best alpha - ElasticNet:", elasticnet_model.alpha_)
print("Best l1_ratio - ElasticNet:", elasticnet_model.l1_ratio_)  # 1.0 = converged to pure Lasso


In [ ]:
# Evaluate all four models on the validation set
results = []
for name, model in [('Linear Regression', lr_model), ('Lasso', lasso_model),
                     ('Ridge', ridge_model), ('ElasticNet', elasticnet_model)]:
    preds = model.predict(X_val)
    r2 = r2_score(y_val, preds)
    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    results.append({'model': name, 'R2': r2, 'MAE': mae, 'RMSE': rmse})

final_results = pd.DataFrame(results)
print(final_results)


## Model Comparison: Linear Regression vs. Lasso vs. Ridge vs. ElasticNet

All four models perform virtually identically (R² ≈ 0.438), consistent with the moderate (not severe) multicollinearity found in the VIF check, and with ElasticNet's l1_ratio converging to 1.0 (pure Lasso). **Lasso** is selected as the final model: same accuracy as the others, but its ability to zero out uninformative features gives a simpler, more interpretable result.

In [ ]:
# Final evaluation on the held-out test set (untouched until now)
lasso_test_preds = lasso_model.predict(X_test)

test_r2 = r2_score(y_test, lasso_test_preds)
test_mae = mean_absolute_error(y_test, lasso_test_preds)
test_rmse = np.sqrt(mean_squared_error(y_test, lasso_test_preds))

print(f"Final Lasso (Test Set): R²={test_r2:.4f}, MAE=€{test_mae:,.2f}, RMSE=€{test_rmse:,.2f}")


In [ ]:
# Coefficients from the unscaled model - useful for the fair 4-way accuracy comparison above,
# but NOT for comparing coefficient *magnitudes* across features (see scaled version below).
coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lasso_model.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print(coef_df)


In [ ]:
# Re-fit on scaled numeric features so coefficient magnitudes are directly comparable
# across brand dummies and continuous features (kilometer/vehicle_age were on very
# different scales, which made the unscaled coefficients above misleading for that purpose).
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = ['kilometer', 'vehicle_age']

X_train_scaled = X_train.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

lasso_scaled = LassoCV(alphas=alphas_wide, cv=5, random_state=42).fit(X_train_scaled, y_train)

coef_df_scaled = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': lasso_scaled.coef_
}).sort_values('coefficient', key=abs, ascending=False)
print(coef_df_scaled)


### Coefficient Interpretation (Scaled)

Mileage and vehicle age both show substantial negative effects on price, on par with mid-tier brand effects rather than negligible as the unscaled coefficients suggested. Brand effects are relative to Audi (the dropped reference category). Notably, Lasso reduced `transmission_type_manual` to exactly zero - genuine feature selection, not just shrinkage, and the concrete evidence for choosing Lasso over Ridge/Linear Regression despite similar overall accuracy.

In [ ]:
def predict_price(brand, kilometer, vehicle_age, transmission_type, unrepaired_damage):
    """Predict a price using the final scaled Lasso model, from raw (unencoded) inputs."""
    new_car = pd.DataFrame({
        'brand_grouped': [brand],
        'kilometer': [kilometer],
        'vehicle_age': [vehicle_age],
        'transmission_type': [transmission_type],
        'unrepaired_damage': [unrepaired_damage]
    })

    # One-hot encode the same way as training
    new_car_encoded = pd.get_dummies(new_car, columns=['brand_grouped', 'transmission_type', 'unrepaired_damage'])

    # Align columns with X_train (adds any missing dummy columns as 0, drops extras, matches order)
    new_car_encoded = new_car_encoded.reindex(columns=X_train.columns, fill_value=0)

    # Scale the numeric columns using the already-fitted scaler (never refit on new data)
    new_car_encoded[numeric_cols] = scaler.transform(new_car_encoded[numeric_cols])

    return lasso_scaled.predict(new_car_encoded)[0]

# Example usage
price = predict_price(
    brand='BMW',
    kilometer=100000,
    vehicle_age=5,
    transmission_type='automatic',
    unrepaired_damage='no'
)
print(f"Predicted price: €{price:,.2f}")


In [ ]:
# Save model artifacts for reuse in the Streamlit app / Excel tool
import joblib

joblib.dump(lasso_scaled, 'lasso_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(list(X_train.columns), 'model_columns.pkl')
joblib.dump(numeric_cols, 'numeric_cols.pkl')

# List of brands the model actually knows, for a UI dropdown
brand_list = sorted(autos['brand_grouped'].unique().tolist())
joblib.dump(brand_list, 'brand_list.pkl')

print("Saved: lasso_model.pkl, scaler.pkl, model_columns.pkl, numeric_cols.pkl, brand_list.pkl")
